# Experimentations
This notebook permits to run experimentations to MLFlow locally

In [1]:
import mlflow
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import os
from mlflow.tracking import MlflowClient
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

# check tracking URI
assert "MLFLOW_TRACKING_URI" in os.environ, "MLFLOW_TRACKING_URI non définie"

experiment_name = "2-xgboost"
seuil_utilisateur = 0.1 # seuil de probabilité
random_state=42

client = MlflowClient()

exp = client.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(experiment_name)
else:
    exp_id = exp.experiment_id

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

# Load prepared data
df = pd.read_csv("../.data/modelisation/app_train.csv")
target_col = "TARGET"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

# Prepare scoring
scoring = {
    "auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# Instantiate model
model = XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=0.8,
    random_state=random_state,
    eval_metric="auc"
)

pipeline = Pipeline([
    ('smote', SMOTE(random_state=random_state, sampling_strategy=0.5)),
    ('clf', model)
])

# Cross-validation on training set (SMOTE inside pipeline -> pas de fuite)
cv_results = cross_validate(
    pipeline,
    X=X_train,
    y=y_train,
    cv=cv_splitter,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

with mlflow.start_run(experiment_id=exp_id):
    # Optional: log notebook if présent (ignore if absent)
    try:
        mlflow.log_artifact("3-experimentation-xgboost.ipynb", artifact_path="notebooks")
    except Exception:
        pass

    # Log CV metrics (moyenne + std)
    for metric_name in scoring.keys():
        key = f"test_{metric_name}"
        if key in cv_results:
            mean_val = np.mean(cv_results[key])
            std_val = np.std(cv_results[key])
            mlflow.log_metric(f"cv_{metric_name}_mean", float(np.round(mean_val, 6)))
            mlflow.log_metric(f"cv_{metric_name}_std", float(np.round(std_val, 6)))

    # Fit final model on the whole training set
    pipeline.fit(X_train, y_train)

    # Predict probabilities on test set using the pipeline (important)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Apply threshold to get binary predictions
    y_pred_utilisateur = (y_proba >= seuil_utilisateur).astype(int)

    # Compute metrics (AUC must être calculé sur probabilités)
    precision_util = precision_score(y_test, y_pred_utilisateur, zero_division=0)
    recall_util = recall_score(y_test, y_pred_utilisateur, zero_division=0)
    f1_util = f1_score(y_test, y_pred_utilisateur, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)

    # Log threshold + metrics
    mlflow.log_metric("probability_threshold", float(np.round(seuil_utilisateur, 3)))
    mlflow.log_metric("precision", float(np.round(precision_util, 3)))
    mlflow.log_metric("recall", float(np.round(recall_util, 3)))
    mlflow.log_metric("f1", float(np.round(f1_util, 3)))
    mlflow.log_metric("auc", float(np.round(roc_auc, 3)))

    # Log fitted model: ici on logge le pipeline complet (SMOTE + XGB)
    try:
        mlflow.sklearn.log_model(pipeline, artifact_path="pipeline_model")
    except Exception:
        # fallback: log uniquement le classifieur si besoin
        try:
            mlflow.sklearn.log_model(pipeline.named_steps['clf'], artifact_path="xgb_model")
        except Exception as e:
            print("Model logging failed:", e)

2025/10/20 10:27:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/20 10:27:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run treasured-auk-34 at: http://mlflow:5000/#/experiments/367982218720826513/runs/ebb931f3c22d4f8bbddb4544af528239
🧪 View experiment at: http://mlflow:5000/#/experiments/367982218720826513
